In [1]:
import os
import random
import shutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import glob
from sklearn.model_selection import train_test_split
import torch
import warnings
warnings.filterwarnings('ignore')

# Install ultralytics if not already installed
try:
    import ultralytics
    print("Ultralytics YOLO already installed")
except ImportError:
    print("Installing ultralytics...")
    os.system("pip install ultralytics")
    import ultralytics

from ultralytics import YOLO
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics version: {ultralytics.__version__}")

Ultralytics YOLO already installed
PyTorch version: 2.6.0+cu118
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4050 Laptop GPU
Ultralytics version: 8.3.101


In [2]:
def load_dataset(data_dir):
    """
    Load dataset from all country directories
    Returns: list of dictionaries with image_path, label_path, and bounding boxes
    """
    dataset = []

    # Get all country directories
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    print(f"Found {len(country_dirs)} country directories: {country_dirs}")

    for country_dir in country_dirs:
        country_path = os.path.join(data_dir, country_dir)
        images_path = os.path.join(country_path, 'images')
        labels_path = os.path.join(country_path, 'labels')

        # Get all image files
        image_files = glob.glob(os.path.join(images_path, '*.jpg'))
        print(f"Found {len(image_files)} images in {country_dir}")

        for image_file in image_files:
            # Get corresponding label file
            base_name = os.path.splitext(os.path.basename(image_file))[0]
            label_file = os.path.join(labels_path, f"{base_name}.txt")

            if os.path.exists(label_file):
                # Read bounding boxes from label file
                bboxes = []
                with open(label_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            parts = line.split()
                            if len(parts) == 5:
                                class_id = int(parts[0])
                                x_center = float(parts[1])
                                y_center = float(parts[2])
                                width = float(parts[3])
                                height = float(parts[4])
                                bboxes.append({
                                    'class_id': class_id,
                                    'x_center': x_center,
                                    'y_center': y_center,
                                    'width': width,
                                    'height': height,
                                })

                dataset.append({
                    'image_path': image_file,
                    'label_path': label_file,
                    'bboxes': bboxes,
                    'country': country_dir
                })

    return dataset

# Load the dataset
data_dir = "data"
dataset = load_dataset(data_dir)
print(f"\nTotal samples loaded: {len(dataset)}")

# Define road damage class names
class_names = {
    0: 'Pothole',
    1: 'Alligator Crack',
    2: 'Transverse Crack',
    3: 'Longitudinal Crack',
}

# Dataset statistics
print("=== Dataset Statistics ===")
country_counts = {}
class_counts = {}
total_bboxes = 0

for sample in dataset:
    country = sample['country']
    country_counts[country] = country_counts.get(country, 0) + 1
    total_bboxes += len(sample['bboxes'])
    
    for bbox in sample['bboxes']:
        class_id = bbox['class_id']
        class_counts[class_id] = class_counts.get(class_id, 0) + 1

print(f"\nSamples by country:")
for country, count in sorted(country_counts.items()):
    print(f"  {country}: {count} samples")

print(f"\nTotal bounding boxes: {total_bboxes}")
print(f"Average bounding boxes per image: {total_bboxes/len(dataset):.2f}")

print(f"\nClass distribution:")
for class_id, count in sorted(class_counts.items()):
    class_name = class_names.get(class_id, f"Class {class_id}")
    percentage = (count / total_bboxes) * 100
    print(f"  {class_name}: {count} ({percentage:.1f}%)")

# Filter samples with bounding boxes
samples_with_bboxes = [sample for sample in dataset if len(sample['bboxes']) > 0]
print(f"\nSamples with bounding boxes: {len(samples_with_bboxes)}")
print(f"Samples without bounding boxes: {len(dataset) - len(samples_with_bboxes)}")

Found 3 country directories: ['country_1', 'country_2', 'country_3']
Found 1976 images in country_1
Found 2082 images in country_2
Found 2082 images in country_2
Found 1981 images in country_3
Found 1981 images in country_3

Total samples loaded: 6039
=== Dataset Statistics ===

Samples by country:
  country_1: 1976 samples
  country_2: 2082 samples
  country_3: 1981 samples

Total bounding boxes: 16238
Average bounding boxes per image: 2.69

Class distribution:
  Pothole: 3425 (21.1%)
  Alligator Crack: 3582 (22.1%)
  Transverse Crack: 4280 (26.4%)
  Longitudinal Crack: 4951 (30.5%)

Samples with bounding boxes: 6039
Samples without bounding boxes: 0

Total samples loaded: 6039
=== Dataset Statistics ===

Samples by country:
  country_1: 1976 samples
  country_2: 2082 samples
  country_3: 1981 samples

Total bounding boxes: 16238
Average bounding boxes per image: 2.69

Class distribution:
  Pothole: 3425 (21.1%)
  Alligator Crack: 3582 (22.1%)
  Transverse Crack: 4280 (26.4%)
  Longit

In [3]:
def prepare_yolo_dataset(dataset, output_dir="yolo_dataset", train_ratio=0.8, val_ratio=0.1):
    """
    Prepare dataset in YOLO format with train/val/test splits
    """
    # Create output directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)
    
    # Split dataset
    train_samples, temp_samples = train_test_split(
        samples_with_bboxes, 
        test_size=1-train_ratio, 
        random_state=42
    )
    
    val_samples, test_samples = train_test_split(
        temp_samples, 
        test_size=val_ratio/(val_ratio + (1-train_ratio-val_ratio)), 
        random_state=42
    )
    
    splits = {
        'train': train_samples,
        'val': val_samples,
        'test': test_samples
    }
    
    print(f"Dataset splits:")
    print(f"  Train: {len(train_samples)} samples")
    print(f"  Val: {len(val_samples)} samples")
    print(f"  Test: {len(test_samples)} samples")
    
    # Copy files to YOLO format
    for split_name, samples in splits.items():
        print(f"\nPreparing {split_name} set...")
        
        for i, sample in enumerate(samples):
            # Copy image
            image_name = os.path.basename(sample['image_path'])
            new_image_path = os.path.join(output_dir, split_name, 'images', image_name)
            shutil.copy2(sample['image_path'], new_image_path)
            
            # Copy label
            label_name = os.path.basename(sample['label_path'])
            new_label_path = os.path.join(output_dir, split_name, 'labels', label_name)
            shutil.copy2(sample['label_path'], new_label_path)
            
            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(samples)} samples")
    
    return splits, output_dir

# Prepare YOLO dataset
print("=== Preparing YOLO Dataset ===")
splits, yolo_dataset_dir = prepare_yolo_dataset(dataset)

# Create YAML config file for YOLO
yaml_config = {
    'path': os.path.abspath(yolo_dataset_dir),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': list(class_names.values())
}

yaml_path = os.path.join(yolo_dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f)

print(f"\nYOLO dataset prepared in: {yolo_dataset_dir}")
print(f"Config file saved as: {yaml_path}")

# Display config
print("\nDataset configuration:")
print(yaml.dump(yaml_config, default_flow_style=False))

=== Preparing YOLO Dataset ===
Dataset splits:
  Train: 4831 samples
  Val: 603 samples
  Test: 605 samples

Preparing train set...
  Processed 100/4831 samples
  Processed 200/4831 samples
  Processed 300/4831 samples
  Processed 400/4831 samples
  Processed 200/4831 samples
  Processed 300/4831 samples
  Processed 400/4831 samples
  Processed 500/4831 samples
  Processed 600/4831 samples
  Processed 700/4831 samples
  Processed 500/4831 samples
  Processed 600/4831 samples
  Processed 700/4831 samples
  Processed 800/4831 samples
  Processed 900/4831 samples
  Processed 1000/4831 samples
  Processed 800/4831 samples
  Processed 900/4831 samples
  Processed 1000/4831 samples
  Processed 1100/4831 samples
  Processed 1200/4831 samples
  Processed 1100/4831 samples
  Processed 1200/4831 samples
  Processed 1300/4831 samples
  Processed 1400/4831 samples
  Processed 1300/4831 samples
  Processed 1400/4831 samples
  Processed 1500/4831 samples
  Processed 1600/4831 samples
  Processed 170

In [4]:
# Initialize YOLO model
print("=== Initializing YOLO Model ===")

# Load pre-trained YOLO model (you can use yolov8n.pt, yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt)
model_size = 'yolov8n.pt'  # Start with nano for faster training, can upgrade to 's', 'm', 'l', 'x'
model = YOLO(model_size)

print(f"Loaded YOLO model: {model_size}")
print(f"Model architecture: {model.model}")

# Training configuration
training_config = {
    'data': yaml_path,
    'epochs': 100,
    'batch': 16,  # Adjust based on your GPU memory
    'imgsz': 640,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'project': 'road_damage_detection',
    'name': 'yolo_finetune',
    'save_period': 10,
    'patience': 20,
    'optimizer': 'AdamW',
    'lr0': 0.01,
    'lrf': 0.1,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'augment': True,
    'mosaic': 1.0,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'degrees': 0.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'resume': False,
    'exist_ok': True,
    'verbose': True
}

print("\nTraining Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Display model summary
print("\n=== Model Summary ===")
print(model.model)

=== Initializing YOLO Model ===


100%|██████████| 6.25M/6.25M [00:00<00:00, 18.8MB/s]

Loaded YOLO model: yolov8n.pt
Model architecture: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=Tr

In [5]:
# Start training
print("=== Starting YOLO Training ===")
print("This may take several hours depending on your hardware...")

# Train the model
results = model.train(**training_config)

print("Training completed!")
print(f"Best model saved at: {results.save_dir}")
print(f"Training results: {results}")

=== Starting YOLO Training ===
This may take several hours depending on your hardware...
New https://pypi.org/project/ultralytics/8.3.168 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.101  Python-3.12.3 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
Ultralytics 8.3.101  Python-3.12.3 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=yolo_dataset\data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=cuda, workers=4, project=road_damage_detection, name=yolo_finetune, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False

100%|██████████| 755k/755k [00:00<00:00, 28.4MB/s]



Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytic

100%|██████████| 5.35M/5.35M [00:00<00:00, 23.3MB/s]



AMP: checks passed 


...

     27/100      2.83G      1.924      2.211      1.779         65        640: 100%|██████████| 302/302 [01:00<00:00,  4.96it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/19 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:05<00:00,  3.29it/s]

                   all        603       1661      0.427      0.396      0.371      0.161



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     28/100      2.85G      1.926      2.181      1.769         61        640:  81%|████████▏ | 246/302 [00:50<00:11,  4.89it/s]



KeyboardInterrupt: 

In [ ]:
# Load the best trained model
print("=== Loading Best Model for Evaluation ===")
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

print(f"Loaded best model from: {best_model_path}")

# Validate the model
print("\n=== Model Validation ===")
validation_results = best_model.val(data=yaml_path, split='val')

print(f"Validation results: {validation_results}")

# Test on test set
print("\n=== Model Testing ===")
test_results = best_model.val(data=yaml_path, split='test')

print(f"Test results: {test_results}")

# Display metrics
print("\n=== Performance Metrics ===")
print(f"mAP@0.5: {validation_results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {validation_results.box.map:.4f}")
print(f"Precision: {validation_results.box.mp:.4f}")
print(f"Recall: {validation_results.box.mr:.4f}")

# Per-class metrics
print("\n=== Per-Class Metrics ===")
for i, class_name in enumerate(class_names.values()):
    if i < len(validation_results.box.ap50):
        ap50 = validation_results.box.ap50[i]
        print(f"{class_name}: AP@0.5 = {ap50:.4f}")
    else:
        print(f"{class_name}: No predictions")

In [ ]:
def visualize_predictions(model, test_images, confidence_threshold=0.5, max_images=6):
    """
    Visualize model predictions on test images
    """
    # Get random test images
    test_image_paths = random.sample(test_images, min(max_images, len(test_images)))
    
    # Calculate grid dimensions
    n_cols = 3
    n_rows = (len(test_image_paths) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 6*n_rows))
    axes = axes.flatten() if len(test_image_paths) > 1 else [axes]
    
    colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'pink', 'brown']
    
    for i, image_path in enumerate(test_image_paths):
        # Make prediction
        results = model.predict(image_path, conf=confidence_threshold)
        
        # Load and display image
        img = Image.open(image_path)
        axes[i].imshow(img)
        
        # Draw predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().numpy()
                cls = int(box.cls[0].cpu().numpy())
                
                # Draw bounding box
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                   fill=False, color=colors[cls % len(colors)], 
                                   linewidth=2)
                axes[i].add_patch(rect)
                
                # Add label
                class_name = list(class_names.values())[cls]
                axes[i].text(x1, y1-10, f'{class_name}: {conf:.2f}',
                           bbox=dict(boxstyle="round,pad=0.3", 
                                   facecolor=colors[cls % len(colors)], 
                                   alpha=0.8),
                           fontsize=10, color='white', fontweight='bold')
        
        axes[i].set_title(f'Predictions: {os.path.basename(image_path)}')
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(len(test_image_paths), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Get test images
test_images = [sample['image_path'] for sample in splits['test']]
print(f"Found {len(test_images)} test images")

# Visualize predictions
print("=== Model Predictions on Test Images ===")
visualize_predictions(best_model, test_images, confidence_threshold=0.3, max_images=6)

In [ ]:
# Visualize training results
print("=== Training Results Visualization ===")

# Plot training curves
results_dir = results.save_dir
results_csv = os.path.join(results_dir, 'results.csv')

if os.path.exists(results_csv):
    # Read training results
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Remove any whitespace
    
    # Create subplots for different metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot losses
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red')
    axes[0, 0].set_title('Box Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='blue')
    axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='red')
    axes[0, 1].set_title('Classification Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Plot metrics
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', color='green')
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='orange')
    axes[1, 0].set_title('Mean Average Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('mAP')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='purple')
    axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='brown')
    axes[1, 1].set_title('Precision and Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    print("\n=== Final Training Metrics ===")
    final_metrics = df.iloc[-1]
    print(f"Final mAP@0.5: {final_metrics['metrics/mAP50(B)']:.4f}")
    print(f"Final mAP@0.5:0.95: {final_metrics['metrics/mAP50-95(B)']:.4f}")
    print(f"Final Precision: {final_metrics['metrics/precision(B)']:.4f}")
    print(f"Final Recall: {final_metrics['metrics/recall(B)']:.4f}")
    
else:
    print("Training results CSV not found. Results may be in a different location.")

# Display confusion matrix if available
confusion_matrix_path = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(confusion_matrix_path):
    print("\n=== Confusion Matrix ===")
    cm_img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(cm_img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()
else:
    print("Confusion matrix not found.")

In [ ]:
# Export model for deployment
print("=== Model Export ===")

# Export to different formats
export_formats = ['onnx', 'torchscript']  # You can add more formats like 'tflite', 'coreml', etc.

for format_type in export_formats:
    try:
        print(f"Exporting to {format_type}...")
        exported_model = best_model.export(format=format_type)
        print(f"Model exported successfully to {format_type}: {exported_model}")
    except Exception as e:
        print(f"Failed to export to {format_type}: {e}")

# Save model summary and configuration
model_info = {
    'model_size': model_size,
    'num_classes': len(class_names),
    'class_names': class_names,
    'training_config': training_config,
    'best_model_path': best_model_path,
    'dataset_path': yolo_dataset_dir,
    'final_metrics': {
        'mAP50': float(validation_results.box.map50),
        'mAP50-95': float(validation_results.box.map),
        'precision': float(validation_results.box.mp),
        'recall': float(validation_results.box.mr)
    }
}

# Save model info
info_path = os.path.join(results.save_dir, 'model_info.yaml')
with open(info_path, 'w') as f:
    yaml.dump(model_info, f, default_flow_style=False)

print(f"Model information saved to: {info_path}")

# Create inference function
def predict_road_damage(image_path, confidence_threshold=0.5):
    """
    Predict road damage on a single image
    """
    results = best_model.predict(image_path, conf=confidence_threshold)
    
    predictions = []
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            predictions.append({
                'class_id': cls,
                'class_name': list(class_names.values())[cls],
                'confidence': float(conf),
                'bbox': [float(x1), float(y1), float(x2), float(y2)]
            })
    
    return predictions

print("\n=== Inference Function Created ===")
print("Use predict_road_damage(image_path) to make predictions on new images")

# Test inference function
test_image = test_images[0]
sample_predictions = predict_road_damage(test_image, confidence_threshold=0.3)

print(f"\nSample prediction on {os.path.basename(test_image)}:")
for pred in sample_predictions:
    print(f"  {pred['class_name']}: {pred['confidence']:.3f} at {pred['bbox']}")

print("\n=== Training Complete ===")
print(f"Best model saved at: {best_model_path}")
print(f"Training results saved at: {results.save_dir}")
print(f"Model can be loaded with: YOLO('{best_model_path}')")